# Student starter
Run cells in order. An incomplete TODO prints guidance and stops subsequent cells. After implementing it, restart the kernel and run again.

# Exercise 6 — Sentence Embeddings and Semantic Similarity

Encode complete texts, inspect N×384 vectors and reuse manual cosine. Preload from the repository root with `python -m src.sentence_resources --download`; ordinary loading is local-only. No paid credentials or GPU. All algorithms live in src; this notebook imports them.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from pathlib import Path
        import sys
        root = Path.cwd()
        if root.name == "notebooks":
            root = root.parent
        if not (root / "src").exists():
            raise RuntimeError("Start the kernel in the repository root or notebooks directory.")
        sys.path.insert(0, str(root))
        from src.sentence_resources import load_sentence_embedding_model, SENTENCE_MODEL_ID
        from src.embeddings import encode_sentences, sentence_similarity, pairwise_tfidf_similarity, SIMILARITY_PAIRS
        from src.classical_nlp import cosine_similarity, build_query_tfidf_vector
        from src.retrieval import tokenize_for_search, prepare_search_corpus
        model = load_sentence_embedding_model()
        print(SENTENCE_MODEL_ID, model.device)

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## One complete text → one vector

Keep original text for the pretrained tokenizer. One sentence stays a two-dimensional batch. Empty list gives (0,384); blank strings are rejected. No extra normalization is requested. The exact model includes Normalize, so its native output already has norm about 1; we retain it and still calculate cosine manually.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        a = "Natural language processing helps computers understand human language."
        b = "Machines can analyze the meaning of human communication."
        c = "Database systems organize structured records."
        one = encode_sentences(model, [a])
        print("Shape:", one.shape, "First ten:", one[0, :10])
        print("Norm:", cosine_similarity(one[0].tolist(), one[0].tolist(), explain=True)["magnitude_a"])
        assert one.shape == (1, 384)

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Batch encoding and cosine reuse

Each row represents an entire text. Predict A–B versus A–C, then measure. Inference/eval mode, no stochastic training. Values can vary slightly across platforms.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        batch = encode_sentences(model, [a, b, c])
        print("Batch shape:", batch.shape)
        related = sentence_similarity(model, a, b)
        unrelated = sentence_similarity(model, a, c)
        print("Related:", related, "Unrelated:", unrelated)
        assert related > unrelated
        assert abs(related - cosine_similarity(batch[0].tolist(), batch[1].tolist())) < 1e-6
        print(cosine_similarity(batch[0].tolist(), batch[1].tolist(), explain=True))

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Lexical overlap versus semantic meaning

The classroom TF-IDF baseline is fitted on eight preset texts plus distinct custom inputs, not merely the two texts (which would give every shared term IDF zero). It uses normalized TF × ln(N/DF). This differs from Task 06 fixed 20-course IDF. A high score does not guarantee factual equivalence: inspect the reversed-role pair.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        comparison = [{"Pair": label, "A": first, "B": second,
                       "TF-IDF": pairwise_tfidf_similarity(first, second),
                       "Sentence embedding": sentence_similarity(model, first, second)}
                      for label, first, second in SIMILARITY_PAIRS]
        for row in comparison:
            print(row)

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Actual Task 06 failure: two manually chosen descriptions

The human-communication query ranked NLP second lexically. Reuse the exact 20-course corpus IDF and compare only NLP/Distributed Systems with MiniLM. No full corpus semantic encoding or ranking.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        import csv
        with (root / "data/search_corpus/course_descriptions.csv").open(encoding="utf-8", newline="") as file:
            courses = list(csv.DictReader(file))
        lexical = prepare_search_corpus([{"document_id": row["course_code"], "text": row["description"]} for row in courses])
        query = "Which subject teaches machines to work with human communication?"
        query_vector = build_query_tfidf_vector(tokenize_for_search(query), lexical["vocabulary"], lexical["idf"])
        failure_case = []
        for name in ["Natural Language Processing", "Distributed Systems"]:
            course = next(row for row in courses if row["course_name"] == name)
            failure_case.append({"Course": name, "Description": course["description"],
                                 "TF-IDF": cosine_similarity(query_vector, lexical["matrix"][courses.index(course)]),
                                 "Sentence embedding": sentence_similarity(model, query, course["description"])})
        for row in failure_case:
            print(row)

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Experiments and limitations

Modify wording; find a paraphrase, unrelated pair and high-overlap/different-meaning pair. Write observations rather than assume embeddings always win. Similarity is not probability, entailment or factual correctness. Word2Vec: static word vector; Sentence Transformer: complete text vector composed from contextual representations, not our Word2Vec average. Inputs exceeding 256 word pieces are truncated. Model setup/cache is starter code; no fallback to simulated vectors. Full attention architecture comes later.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        custom_a = "Students study natural language processing."
        custom_b = "Students learn natural language processing."
        print("Custom TF-IDF:", pairwise_tfidf_similarity(custom_a, custom_b))
        print("Custom sentence cosine:", sentence_similarity(model, custom_a, custom_b))
        print("Empty batch:", encode_sentences(model, []).shape)

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Transition

We can now represent a question and a course description in the same semantic vector space. What if we compared the query with every course and ranked them? Semantic Search is the next task, not implemented here.

Exercise 6 Checkpoint 5 — Semantic Sentence Similarity Works (separate from the original roadmap Transformer checkpoint). Suggested commit: `Add sentence embedding similarity`.